In [1]:
import torch as t

In [2]:
#Auto_grad的使用方式
a = t.tensor([1.,2.,3.],requires_grad=True)#首先必须要是浮点数

b = a.mul(a)
loss = b.sum()#多个变量的时候需要求和

loss.backward()

print(a.grad)


tensor([2., 4., 6.])


In [3]:
import torch.nn as nn

In [4]:
a = t.rand(3,4)
b = t.ones((4,1),requires_grad=True)
c = a.mm(b)
print(c)
print(c.shape)
print(c.requires_grad)

tensor([[1.7743],
        [2.2590],
        [2.1686]], grad_fn=<MmBackward0>)
torch.Size([3, 1])
True


In [5]:
#使用nn.Parameter并查看nn.Parameter内部常用函数的调用方式
class Test_Parameter(nn.Module):
    def __init__(self,in_dim):
        super().__init__()
        self.w = nn.Parameter(t.rand(in_dim+1,1))
        self.w1 = nn.Parameter(t.tensor([[1.],[3.]]))
    
    def show_infro(self,flag):
        if flag:
            for name, parameter in self.named_parameters():
                print(name,'\n',parameter,'\n',parameter.data)
        else:
            for parameter in self.parameters():
                print(parameter,'\n',parameter.data)

In [6]:
a = Test_Parameter(3)
a.show_infro(True)
print('-------------------')
a.show_infro(False)

w 
 Parameter containing:
tensor([[0.7607],
        [0.0714],
        [0.8265],
        [0.4237]], requires_grad=True) 
 tensor([[0.7607],
        [0.0714],
        [0.8265],
        [0.4237]])
w1 
 Parameter containing:
tensor([[1.],
        [3.]], requires_grad=True) 
 tensor([[1.],
        [3.]])
-------------------
Parameter containing:
tensor([[0.7607],
        [0.0714],
        [0.8265],
        [0.4237]], requires_grad=True) 
 tensor([[0.7607],
        [0.0714],
        [0.8265],
        [0.4237]])
Parameter containing:
tensor([[1.],
        [3.]], requires_grad=True) 
 tensor([[1.],
        [3.]])


In [9]:
w = t.ones(4,3,requires_grad=True)
x = t.rand(3,1,requires_grad=True)
f = sum(w@x+b)
f.backward()
print(x.requires_grad)
print(w.requires_grad)
print(x.grad)

True
True
tensor([[4.],
        [4.],
        [4.]])


In [13]:
import torch.nn.functional as F

In [19]:
w = t.rand(4,3,requires_grad=True)
x = t.rand(3,1,requires_grad=True)
b = t.zeros(4,1,requires_grad=True)

a = w@x
output = F.relu(a+b)
loss = sum(output)

loss.backward()

print(x.grad)
print(w.grad)
print(b.grad)
print(x)
print(output.grad)
print(a.grad)

tensor([[1.4153],
        [2.5284],
        [2.3836]])
tensor([[0.5735, 0.8139, 0.2806],
        [0.5735, 0.8139, 0.2806],
        [0.5735, 0.8139, 0.2806],
        [0.5735, 0.8139, 0.2806]])
tensor([[1.],
        [1.],
        [1.],
        [1.]])
tensor([[0.5735],
        [0.8139],
        [0.2806]], requires_grad=True)
None
None


C:\Users\L\AppData\Local\Temp\ipykernel_34160\611181323.py:15: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:499.)
  print(output.grad)
C:\Users\L\AppData\Local\Temp\ipykernel_34160\611181323.py:16: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mis

In [ ]:
#先回顾numpy如何计算线性回归的
import numpy as np

class MultiLinearRegression(object):
    def __init__ (self,dim_in,lr,max_iter,seed):
        np.random.seed(seed)
        self.W = np.random.normal(loc = 1, scale = 0.1, size = (dim_in+1,1))
        self.lr = lr
        self.max_iter = max_iter
        self.loss_arr = []
    
    def train(self,X,y):
        loss_list = []
        X = np.hstack([X,np.ones((X.shape[0],1))])
        for i in range(self.max_iter):
            dw = self.cal_grad(X,y) #还收缩了train函数，把fit和train融合到一起来了
            self.W = self.W - dw * self.lr
            y_pred = X @ self.W #这里收缩了predict函数
            loss = self.cal_loss(y_pred,y)
            loss_list.append(loss)
            self.loss_arr = loss_list
        return self.loss_arr

    def cal_loss(self,y_pred,y):
        return np.mean((y-y_pred)**2)

    def cal_grad(self,X,y):
        N = X.shape[0] - 1
        diff = X @ self.W - y
        grad = X.T @ diff
        dw = (2 * grad) / N
        return dw

In [23]:
# data generation
np.random.seed(272)
data_size = 100

dim_in = 3
dim_out = 1

x = np.random.uniform(low=1.0, high=10.0, size=[data_size, dim_in])
print("x = ", x)
print("x.shape = ", x.shape)
w_true = np.array([[1.5], [-5.], [3.]])
y = np.matmul(x, w_true)  + np.random.normal(loc=0.0, scale=10.0, size=[data_size, dim_out])

x =  [[3.22948482 8.90713648 9.44108164]
 [3.7540866  5.18814828 2.56289109]
 [3.00154469 3.54680345 6.56105895]
 [2.77558845 8.7331636  4.55623757]
 [7.27448618 5.21980683 2.02123466]
 [5.70244546 9.04862078 3.61807215]
 [8.26495323 1.17483877 7.93409977]
 [6.80274505 6.44082128 5.39565557]
 [7.87504622 3.18006603 9.74028186]
 [7.85282943 1.46440793 1.67410887]
 [3.98889502 3.87154281 2.65275974]
 [5.18012949 7.58711931 9.29960965]
 [1.04624586 1.15467664 3.25266916]
 [5.2879521  1.93090899 6.34184487]
 [3.48792901 6.82321567 1.86626583]
 [7.24899019 1.53388668 7.94965088]
 [5.09122067 3.85023415 6.99344605]
 [7.41953419 5.62320728 2.76467079]
 [9.55238812 4.24228745 6.29956213]
 [9.69569912 7.98976674 9.9880322 ]
 [2.19406312 4.7533448  8.44340561]
 [4.84655189 8.48405352 6.40979505]
 [5.91926314 6.3675922  7.53056809]
 [6.49461497 5.95364464 9.78171215]
 [7.32064404 3.21811252 7.01256725]
 [2.22014269 6.71627606 7.64192303]
 [4.66348509 2.21697786 3.19788601]
 [9.48460207 9.43819064

In [24]:
# train / test split
shuffled_index = np.random.permutation(data_size)
x = x[shuffled_index, :]
y = y[shuffled_index, :]


split_index = int(data_size * 0.7)
x_train = x[:split_index, :]
y_train = y[:split_index, :]
x_test = x[split_index:, :]
y_test = y[split_index:, :]

In [25]:
# train the liner regression model
regr = MultiLinearRegression(dim_in, lr=0.01, max_iter=100, seed=0)
regr.train(x_train, y_train)
print(regr.W)

[[ 0.52992403]
 [-4.91098013]
 [ 3.3051611 ]
 [ 0.93722341]]


In [27]:
import matplotlib.pyplot as plt
#下面是在为画图做准备
x_test_aug = np.hstack([x_test, np.ones((x_test.shape[0], 1))])
y_pred = x_test_aug @ regr.W
res = y_pred - y_test
plt.scatter(np.arange(len(regr.loss_arr)), regr.loss_arr, marker='o', c='green')
plt.savefig('output-practice.png')  # 代替 plt.show()
plt.close()

In [31]:
import torch as t
import nn.torch as nn
#Parameter
class TestParameter(nn.Module):#继承nn.Module
    def __init__(self, in_dim): #构造函数，需要调用nn.Mudule的构造函数
        super().__init__()       #等价于nn.Module.__init__()
        #使用parameter类
        self.w = nn.Parameter(torch.randn(in_dim+1, 1))
        #未使用parameter类；一个module类可以挂载多个对象
        self.w1 = torch.randn(in_dim + 1, 1)      
        self.w2 = nn.Parameter(torch.randn(in_dim+1, 1))
        #self.w1 = nn.Parameter(torch.randn(in_dim + 1, 1))
        #self.w2 = nn.Parameter(torch.randn(in_dim + 1, 1))
        
        print("w.size()=", self.w.size())

    def show_info(self, flag): 
        if flag:
            for name, parameter in self.named_parameters():
                print(name, '\n', parameter, '\n', parameter.data)
        else:
            for parameter in self.parameters():
                print(parameter, '\n', parameter.data)

OSError: [WinError 127] 找不到指定的程序。 Error loading "d:\Ruanjian\Anaconda_envs\envs\env1-py38\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [28]:
test = TestParameter(4)
test.show_info(True)
test.show_info(False)

NameError: name 'TestParameter' is not defined